In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split



In [2]:
df = pd.read_csv('insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
print("shape:", df.shape)
df.info()

shape: (1338, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 94.5 KB


In [4]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [5]:
#انا معنديش قيم مفقوده بس هعمل الخطوه دي احتياطي

numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mean())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("remaining nulls:")
df.isnull().sum()

remaining nulls:


C:\Users\wafdy\AppData\Local\Temp\ipykernel_40608\2743963419.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [6]:
print("number of duplicated rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("shape after removing duplicates:", df.shape)

number of duplicated rows: 1
shape after removing duplicates: (1337, 7)


In [21]:
X = df.drop('charges', axis=1)
y = df[['charges']]   
print("X columns:", list(X.columns))

X columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'region']


In [22]:
categorical_cols = ['sex', 'smoker', 'region']

for col in categorical_cols:
    X[col] = pd.Categorical(X[col], categories=X[col].unique())

X = pd.get_dummies(X, columns=categorical_cols, drop_first=False)

dummy_cols = [c for c in X.columns if c.startswith(('sex_', 'smoker_', 'region_'))]
X[dummy_cols] = X[dummy_cols].astype(int)

sex_cols = [c for c in X.columns if c.startswith('sex_')]
smoker_cols = [c for c in X.columns if c.startswith('smoker_')]
region_cols = [c for c in X.columns if c.startswith('region_')]

rename_map = {}
for i, col in enumerate(sex_cols, start=1):
    rename_map[col] = f'sex_{i}'
for i, col in enumerate(smoker_cols, start=1):
    rename_map[col] = f'smoker_{i}'
for i, col in enumerate(region_cols, start=1):
    rename_map[col] = f'region_{i}'

X = X.rename(columns=rename_map)


X.head()

,age,bmi,children,sex_1,sex_2,smoker_1,smoker_2,region_1,region_2,region_3,region_4
0,19,27.900,0,1,0,1,0,1,0,0,0
1,18,33.770,1,0,1,0,1,0,1,0,0
2,28,33.000,3,0,1,0,1,0,1,0,0
3,33,22.705,0,0,1,0,1,0,0,1,0
4,32,28.880,0,0,1,0,1,0,0,1,0


In [23]:
scaler = MinMaxScaler()
cols_to_scale = ['age', 'bmi', 'children']
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

X.head()

,age,bmi,children,sex_1,sex_2,smoker_1,smoker_2,region_1,region_2,region_3,region_4
0,0.021739,0.321227,0.0,1,0,1,0,1,0,0,0
1,0.000000,0.479150,0.2,0,1,0,1,0,1,0,0
2,0.217391,0.458434,0.6,0,1,0,1,0,1,0,0
3,0.326087,0.181464,0.0,0,1,0,1,0,0,1,0
4,0.304348,0.347592,0.0,0,1,0,1,0,0,1,0


In [24]:
y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y)
y = pd.DataFrame(y, columns=["charges"])

y

,charges
0,0.251611
1,0.009636
2,0.053115
3,0.333010
4,0.043816
...,...
1332,0.151299
1333,0.017305
1334,0.008108
1335,0.014144


In [25]:
print(X.dtypes)
print(y.dtypes)


age         float64
bmi         float64
children    float64
sex_1         int64
sex_2         int64
smoker_1      int64
smoker_2      int64
region_1      int64
region_2      int64
region_3      int64
region_4      int64
dtype: object
charges    float64
dtype: object


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1069, 11)
X_test shape: (268, 11)
y_train shape: (1069, 1)
y_test shape: (268, 1)
